# Monthly Retail Revenue Forecasting

**Project 3 of 10** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **monthly revenue** for a UK-based online retailer using the UCI Online Retail dataset hosted on Kaggle. We aggregate transactional data to monthly totals and apply statistical forecasting models.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting |
| **Target variable** | `Revenue` (monthly total) |
| **Date column** | `InvoiceDate` |
| **Frequency** | Monthly (`MS`) |
| **Seasonal period** | 12 (annual cycle) |
| **Primary TS library** | StatsForecast |
| **Kaggle dataset** | `carrie1/ecommerce-data` |

## Learning Objectives

By completing this notebook you will learn how to:

1. Download and parse a **transactional retail dataset** with invoice-level detail
2. **Derive revenue** from `Quantity * UnitPrice` and handle cancellations/returns
3. **Aggregate to monthly frequency** from raw transaction timestamps
4. Explore **trend and annual seasonality** in monthly revenue data
5. Handle the challenge of **short monthly series** (~13 data points)
6. Build naive and seasonal-naive baselines
7. Benchmark regressors via LazyPredict on lag-feature tabular view
8. Run FLAML AutoML with a time budget
9. Train StatsForecast models (AutoARIMA, AutoETS, AutoTheta)
10. Evaluate with MAE, RMSE, MAPE and compare all approaches

## Problem Statement

Given ~13 months of transactional e-commerce data from a UK-based online retailer, **forecast monthly revenue for the next 3 months**.

The data contains individual invoice lines. We must aggregate to monthly totals, handle returns (negative quantities), and build a forecast despite the relatively short history.

## Why This Project Matters

- **Cash flow planning**: Monthly revenue forecasts drive budgeting, investment, and hiring decisions.
- **Inventory procurement**: Monthly demand signals determine bulk purchasing schedules.
- **Growth tracking**: Understanding revenue trend and seasonality helps set realistic growth targets.
- **Short-series challenge**: Many real businesses have limited history — learning to forecast with short monthly series is a critical skill.

## Dataset Overview

| Attribute | Value |
|-----------|-------|
| **Source** | UCI ML Repository (hosted on Kaggle) |
| **File** | Single CSV with ~541K transaction rows |
| **Date range** | Dec 2010 — Dec 2011 (~13 months) |

### Key columns
- **InvoiceNo** — invoice number (prefix 'C' = cancellation)
- **Quantity** — units ordered (negative for returns)
- **UnitPrice** — price per unit in GBP
- **InvoiceDate** — timestamp
- **CustomerID** — customer identifier (some NaN)
- **Country** — customer country (mostly UK)

## Dataset Source & License Notes

- **Kaggle dataset**: [E-Commerce Data](https://www.kaggle.com/datasets/carrie1/ecommerce-data)
- **Original source**: UCI Machine Learning Repository
- **License**: CC BY-SA 4.0
- **Usage**: Educational purposes only

## Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "statsforecast", "lightgbm",
    "statsmodels", "scipy",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Monthly Retail Revenue Forecasting"
KAGGLE_SLUG  = "carrie1/ecommerce-data"

TARGET  = "Revenue"
DATE    = "InvoiceDate"
FREQ    = "MS"

SEASON_LENGTH    = 12
FORECAST_HORIZON = 3
TEST_SIZE        = FORECAST_HORIZON
VAL_SIZE         = 2
RANDOM_STATE     = 42
FLAML_BUDGET     = 60

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} months")

## Kaggle Credential Check

Before downloading, we verify that Kaggle credentials are available.

In [ ]:
kaggle_ok = False

if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True

kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    raise RuntimeError(
        "No Kaggle credentials found!\n"
        "Set KAGGLE_API_TOKEN env var, or place kaggle.json in ~/.kaggle/\n"
        "See: https://www.kaggle.com/settings -> Create New Token"
    )
print("Ready to download.")

## Dataset Download & Loading

We download the Online Retail dataset from Kaggle using `kagglehub`.

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    print("Falling back to kaggle CLI...")
    os.makedirs("data", exist_ok=True)
    ret = os.system(f'kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip')
    if ret != 0:
        raise RuntimeError("Both kagglehub and kaggle CLI failed.")
    data_path = pathlib.Path("data")

csv_files = sorted(data_path.rglob("*.csv"))
for f in csv_files:
    print(f"  {f.name:35s}  {f.stat().st_size / 1e6:7.2f} MB")
assert len(csv_files) > 0, "No CSV files found after download!"

In [ ]:
raw = pd.read_csv(csv_files[0], encoding="ISO-8859-1", parse_dates=["InvoiceDate"])
print(f"Raw data: {raw.shape}")
raw.head()

## Data Validation Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)
assert "InvoiceDate" in raw.columns, "Missing InvoiceDate!"
assert "Quantity" in raw.columns, "Missing Quantity!"
assert "UnitPrice" in raw.columns, "Missing UnitPrice!"
print(f"  Shape          : {raw.shape[0]:,} rows x {raw.shape[1]} cols")
print(f"  Date range     : {raw['InvoiceDate'].min()} to {raw['InvoiceDate'].max()}")
for col in raw.columns:
    n_miss = raw[col].isna().sum()
    if n_miss > 0:
        print(f"  {col} missing: {n_miss:,} ({n_miss/len(raw)*100:.1f}%)")
print(f"  Negative Qty   : {(raw['Quantity'] < 0).sum():,}")
print(f"  Cancellations  : {raw['InvoiceNo'].astype(str).str.startswith('C').sum():,}")
print(f"  Countries      : {raw['Country'].nunique()}")
print(f"  Duplicates     : {raw.duplicated().sum():,}")
print("\nValidation complete.")

## Data Cleaning / Preprocessing

We remove cancellations, filter out invalid prices, compute Revenue, and aggregate to monthly totals.

In [ ]:
df = raw.copy()
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]  # remove cancellations
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df = df.dropna(subset=["CustomerID"])
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
print(f"After cleaning: {len(df):,} rows (removed {len(raw)-len(df):,})")

# Aggregate to monthly revenue
monthly = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))
    .agg(Revenue=("Revenue", "sum"), Transactions=("InvoiceNo", "nunique"))
    .reset_index()
)
monthly["InvoiceDate"] = monthly["InvoiceDate"].dt.to_timestamp()
monthly = monthly.sort_values("InvoiceDate").reset_index(drop=True)
print(f"\nMonthly series: {len(monthly)} months")
print(f"Date range: {monthly['InvoiceDate'].min().date()} to {monthly['InvoiceDate'].max().date()}")
monthly

## Exploratory Data Analysis

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly["InvoiceDate"], y=monthly["Revenue"],
                         name="Monthly Revenue", mode="lines+markers",
                         line=dict(color="blue", width=2)))
fig.update_layout(title="Monthly Revenue — Online Retail", template="plotly_white",
                  xaxis_title="Month", yaxis_title="Revenue (GBP)")
fig.show()

In [ ]:
monthly["MonthNum"] = monthly["InvoiceDate"].dt.month
fig = px.bar(monthly, x="MonthNum", y="Revenue", title="Revenue by Month of Year")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Trend visualisation
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly["InvoiceDate"], monthly["Revenue"], marker="o", label="Revenue")
ax.plot(monthly["InvoiceDate"], monthly["Revenue"].rolling(3, center=True).mean(),
        label="3-month MA", color="red", linewidth=2)
ax.set_title("Monthly Revenue with 3-month Moving Average")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Stationarity test
adf = adfuller(monthly["Revenue"].values)
print("Augmented Dickey-Fuller Test:")
print(f"  ADF Statistic : {adf[0]:.4f}")
print(f"  p-value       : {adf[1]:.6f}")
print(f"  Result: {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'} at 5%")
print(f"\nNote: With only {len(monthly)} observations, ADF has low power.")

## Target Analysis

In [ ]:
print("Target Statistics (monthly revenue):")
print(monthly["Revenue"].describe().to_string())
print(f"\nSkewness : {monthly['Revenue'].skew():.3f}")
print(f"Kurtosis : {monthly['Revenue'].kurtosis():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(monthly["Revenue"], bins=10, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Monthly Revenue")
axes[1].boxplot(monthly["Revenue"].dropna())
axes[1].set_title("Box Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

With ~13 months of data we use a tight temporal split. Results should be interpreted cautiously due to the short series.

In [ ]:
ts_df = monthly[["InvoiceDate", "Revenue"]].rename(
    columns={"InvoiceDate": "ds", "Revenue": "y"}).copy()

n = len(ts_df)
if n > TEST_SIZE + VAL_SIZE + 3:
    ts_train = ts_df.iloc[: n - TEST_SIZE - VAL_SIZE].copy()
    ts_val   = ts_df.iloc[n - TEST_SIZE - VAL_SIZE : n - TEST_SIZE].copy()
    ts_test  = ts_df.iloc[n - TEST_SIZE :].copy()
else:
    VAL_SIZE = 0
    ts_train = ts_df.iloc[: n - TEST_SIZE].copy()
    ts_val   = pd.DataFrame(columns=ts_df.columns)
    ts_test  = ts_df.iloc[n - TEST_SIZE :].copy()

print(f"Train : {len(ts_train)} months")
if len(ts_val): print(f"Val   : {len(ts_val)} months")
print(f"Test  : {len(ts_test)} months")

ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train", mode="lines+markers"))
if len(ts_val):
    fig.add_trace(go.Scatter(x=ts_val["ds"], y=ts_val["y"], name="Val", mode="lines+markers"))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Test",
                         mode="lines+markers", line=dict(color="red")))
fig.update_layout(title="Temporal Split", template="plotly_white")
fig.show()

## Feature Engineering

For tabular approaches we create lag and calendar features. Monthly data limits available lags.

In [ ]:
def make_features(df):
    out = df.copy()
    for lag in [1, 2, 3]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    out["roll_mean_3"] = out["y"].shift(1).rolling(3).mean()
    out["month"] = out["ds"].dt.month
    out["quarter"] = out["ds"].dt.quarter
    return out

feat_full = make_features(ts_df).dropna().reset_index(drop=True)
feature_cols = [c for c in feat_full.columns if c not in ["ds", "y"]]
print(f"Feature matrix: {feat_full.shape}")

if len(feat_full) > TEST_SIZE + 2:
    train_cut = ts_train["ds"].max() if len(ts_train) else feat_full["ds"].iloc[0]
    val_cut = ts_val["ds"].max() if len(ts_val) else train_cut
    ft = feat_full[feat_full["ds"] <= train_cut]
    fv = feat_full[(feat_full["ds"] > train_cut) & (feat_full["ds"] <= val_cut)] if len(ts_val) else pd.DataFrame()
    fte = feat_full[feat_full["ds"] > val_cut]
    X_train, y_train = ft[feature_cols], ft["y"]
    X_val = fv[feature_cols] if len(fv) else X_train.iloc[-1:]
    y_val = fv["y"] if len(fv) else y_train.iloc[-1:]
    X_test, y_test_f = fte[feature_cols], fte["y"]
    X_tv = pd.concat([X_train, X_val] if len(fv) else [X_train])
    y_tv = pd.concat([y_train, y_val] if len(fv) else [y_train])
    print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
else:
    X_train = X_val = X_test = X_tv = None
    print("Not enough data for lag-feature split.")

## Baseline Approaches

- **Naive**: predict the last known value
- **Seasonal Naive**: predict value from 12 months ago (if available)
- **Moving Average**: predict average of the last 3 months

In [ ]:
def calc_metrics(actual, predicted, name="Model"):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.any() else np.nan
    return {"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2)}

results = []
actual_test = ts_test["y"].values

naive_pred = np.full(TEST_SIZE, ts_trainval["y"].iloc[-1])
results.append(calc_metrics(pd.Series(actual_test), pd.Series(naive_pred), "Naive"))

if len(ts_trainval) >= SEASON_LENGTH:
    sn = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
    sn_pred = np.tile(sn, 2)[:TEST_SIZE]
    results.append(calc_metrics(pd.Series(actual_test), pd.Series(sn_pred), "Seasonal Naive (12)"))

ma_pred = np.full(TEST_SIZE, ts_trainval["y"].iloc[-3:].mean())
results.append(calc_metrics(pd.Series(actual_test), pd.Series(ma_pred), "Moving Avg (3)"))

print("Baseline Results:")
print(pd.DataFrame(results).to_string(index=False))

## LazyPredict Benchmark (Lag-Feature Tabular View)

**Note**: With very few monthly samples LazyPredict results should be interpreted cautiously.

In [ ]:
if X_train is not None and len(X_train) >= 8:
    try:
        lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=RANDOM_STATE)
        lm, _ = lazy.fit(X_train, X_val, y_train, y_val)
        print("LazyPredict top 10:")
        print(lm.head(10).to_string())
        for i, (nm, row) in enumerate(lm.head(3).iterrows()):
            results.append({"Model": f"LP:{nm}", "MAE": round(row.get("MAE",0),2),
                            "RMSE": round(row.get("RMSE",0),2), "MAPE": np.nan})
    except Exception as e:
        print(f"LazyPredict failed: {e}")
else:
    print("Skipping LazyPredict — too few training rows.")

## FLAML AutoML

In [ ]:
if X_tv is not None and len(X_tv) >= 6:
    try:
        flaml_model = AutoML()
        flaml_model.fit(X_train=X_tv, y_train=y_tv, task="regression",
                        time_budget=FLAML_BUDGET, metric="rmse", verbose=0, seed=RANDOM_STATE)
        if len(X_test):
            fp = flaml_model.predict(X_test)
            results.append(calc_metrics(pd.Series(actual_test[:len(fp)]),
                                        pd.Series(fp), f"FLAML ({flaml_model.best_estimator})"))
            print(f"FLAML best: {flaml_model.best_estimator}")
    except Exception as e:
        print(f"FLAML failed: {e}")
else:
    print("Skipping FLAML — too few rows.")

## StatsForecast — Dedicated Time-Series Models

**Why StatsForecast?** For short monthly series, statistical models (ARIMA, ETS, Theta) are often more appropriate than ML. They require fewer data points, handle trend/seasonality natively, and resist overfitting on small samples.

In [ ]:
sf_train = ts_trainval.copy()
sf_train["unique_id"] = "revenue"

season = min(SEASON_LENGTH, len(sf_train) // 2)

sf = StatsForecast(
    models=[
        AutoARIMA(season_length=season),
        AutoETS(season_length=season),
        AutoTheta(season_length=season),
    ],
    freq="MS",
    n_jobs=1,
)
sf.fit(sf_train)
sf_preds = sf.predict(h=FORECAST_HORIZON)
print(f"StatsForecast predictions:")
sf_preds

In [ ]:
for mn in ["AutoARIMA", "AutoETS", "AutoTheta"]:
    if mn in sf_preds.columns:
        pv = sf_preds[mn].values[:TEST_SIZE]
        r = calc_metrics(pd.Series(actual_test), pd.Series(pv), f"SF:{mn}")
        results.append(r)
        print(f"{mn}: MAE={r['MAE']}, RMSE={r['RMSE']}, MAPE={r['MAPE']}%")

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("All Models Ranked by RMSE:")
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print(f"\n{'='*50}\nTOP 3 MODELS:\n{'='*50}")
print(top3.to_string(index=False))

fig = px.bar(results_df, x="Model", y="RMSE", title="Model Comparison — RMSE",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

## Final Evaluation — Forecast vs Actual

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_trainval["ds"], y=ts_trainval["y"], name="Train",
                         mode="lines+markers", line=dict(color="blue")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=actual_test, name="Actual",
                         mode="lines+markers", line=dict(color="black", width=3)))
colors = ["green", "orange", "purple"]
for i, mn in enumerate(["AutoARIMA", "AutoETS", "AutoTheta"]):
    if mn in sf_preds.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=sf_preds[mn].values[:TEST_SIZE],
                                 name=f"SF:{mn}", mode="lines+markers",
                                 line=dict(color=colors[i], dash="dash")))
fig.update_layout(title="Forecast vs Actual — Test Period", template="plotly_white")
fig.show()

## Error Analysis

In [ ]:
best_col = "AutoETS" if "AutoETS" in sf_preds.columns else sf_preds.columns[-1]
best_pred = sf_preds[best_col].values[:TEST_SIZE]
errors = actual_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, TEST_SIZE+1), errors,
            color=["green" if e > 0 else "red" for e in errors])
axes[0].set_title("Forecast Error by Month"); axes[0].axhline(0, color="black", ls="--")
axes[1].scatter(actual_test, best_pred, s=100)
mn, mx = min(actual_test.min(), best_pred.min()), max(actual_test.max(), best_pred.max())
axes[1].plot([mn, mx], [mn, mx], "r--", label="Perfect")
axes[1].set_title("Actual vs Predicted"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"Mean Error (bias): {errors.mean():,.0f} GBP")
print(f"MAE: {np.abs(errors).mean():,.0f} GBP")

## Interpretation & Insights

### Key Findings
1. **Short series challenge**: With ~13 months, statistical models (ARIMA, ETS) often outperform ML approaches.
2. **Seasonal pattern**: Revenue tends to peak in Q4 (Nov-Dec) due to holiday shopping.
3. **Trend**: The UK-based retailer shows growing revenue over the observation period.
4. **Baseline strength**: Simple baselines are competitive on very short series.

## Limitations

1. **Very short history**: Only ~13 months severely limits model complexity and seasonal estimation.
2. **Single country dominance**: ~90% of transactions are UK-based.
3. **No external regressors**: Holidays, marketing spend, economic indicators not included.
4. **Cancellation handling**: We removed all cancellations; partial ones may distort revenue.
5. **Monthly aggregation**: Loses within-month patterns.

## How to Improve This Project

1. **Extend data history**: Obtain more months for better seasonal estimation.
2. **Add covariates**: Marketing spend, holiday calendars, GDP indicators.
3. **Country-level forecasting**: Forecast UK and non-UK separately.
4. **Weekly frequency**: More data points per year.
5. **Hierarchical forecasting**: Forecast by product category and reconcile.

## Production Considerations

1. **Monthly retraining**: Update the model each month.
2. **Confidence intervals**: Use prediction intervals for budget planning.
3. **Monitoring**: Track MAPE monthly.
4. **Revenue recognition**: Align with accounting standards.
5. **Multiple horizons**: 3/6/12-month forecasts for different planning needs.

## Common Mistakes to Avoid

1. **Using ML on very short series**: Prefer statistical models for <50 observations.
2. **Ignoring cancellations**: Not filtering returns inflates then drops revenue unpredictably.
3. **Annual seasonality with <2 years**: Cannot reliably estimate 12-month seasonality.
4. **Random splits**: Always use temporal splits for time series.

## Mini Challenge / Exercises

1. **Weekly frequency**: Aggregate to weekly. Does the extra resolution help?
2. **Country filter**: Forecast UK-only revenue separately.
3. **Log transform**: Apply log transform. Does it improve MAPE?
4. **Prediction intervals**: Use StatsForecast's built-in intervals.
5. **Product category**: Forecast top 5 categories separately.

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Online Retail** dataset and computed revenue from transactions
- Aggregated to monthly totals and validated data quality
- Built baselines (naive, seasonal naive, moving average)
- Attempted LazyPredict and FLAML on lag features
- Trained StatsForecast models (AutoARIMA, AutoETS, AutoTheta)
- Selected top 3 models and analyzed errors

### Key Takeaways
1. **Statistical models excel on short series** — AutoARIMA/ETS/Theta are designed for this
2. **Monthly aggregation trades resolution for simplicity**
3. **Q4 holiday seasonality** is the dominant pattern in retail revenue
4. **Data quality matters** — cancellation handling significantly affects totals

---
*Notebook generated as part of a time-series forecasting portfolio.*
*For educational purposes only.*